In [1]:
# Limpar ambiente (não necessário em Python como em R)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA

from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

# rpy2 para rodar NbClust do R
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

ModuleNotFoundError: No module named 'yellowbrick'

In [2]:
# -------------------------
# Gerar dados simulados
# -------------------------
np.random.seed(10)

num_clientes = 500

consumo_laticinios = np.abs(np.random.normal(loc=50, scale=30, size=num_clientes))
consumo_carne = np.abs(np.random.normal(loc=70, scale=40, size=num_clientes))
consumo_padaria = np.abs(np.random.normal(loc=30, scale=10, size=num_clientes))

dados = pd.DataFrame({
    "laticinios": consumo_laticinios,
    "carne": consumo_carne,
    "padaria": consumo_padaria
})

print(dados.head())

   laticinios       carne    padaria
0   89.947595  127.570149  45.132522
1   71.458369  125.285865  28.576664
2    3.637991  106.852817  28.605800
3   49.748485   47.084511  41.755075
4   68.640079  121.001964  41.837890


In [3]:
# -------------------------
# Padronizar dados
# -------------------------
scaler = StandardScaler()
dados_padronizados = scaler.fit_transform(dados)

In [4]:
# -------------------------
# NbClust (via R)
# -------------------------
utils = importr("utils")

try:
    nbclust = importr("NbClust")
except:
    utils.install_packages("NbClust")
    nbclust = importr("NbClust")

dados_padronizados_df = pd.DataFrame(
    dados_padronizados,
    columns=dados.columns
)

NameError: name 'importr' is not defined

In [5]:
# converter Python -> R
with localconverter(ro.default_converter + pandas2ri.converter):
    r_dados = ro.conversion.py2rpy(dados_padronizados_df)


NameError: name 'localconverter' is not defined

In [6]:
# rodar NbClust
resultado = nbclust.NbClust(
    r_dados,
    distance="euclidean",
    min_nc=3,
    max_nc=10,
    method="kmeans"
)

NameError: name 'nbclust' is not defined

In [7]:
# votos dos índices
best_nc = np.array(resultado.rx2("Best.nc"))
k_votes = best_nc[0]

freq = pd.Series(k_votes).value_counts().sort_index()

NameError: name 'resultado' is not defined

In [8]:
# gráfico NbClust
plt.figure(figsize=(8,6))
plt.bar(freq.index, freq.values, color="steelblue")

plt.xlabel("Número de clusters k")
plt.ylabel("Frequência entre índices")
plt.title(f"Número ótimo de clusters - k = {freq.idxmax()}")

plt.xticks(range(2,11))
plt.show()

# k ideal segundo NbClust
k_ideal = int(freq.idxmax())

print(f"Número ótimo de clusters segundo NbClust: {k_ideal}")

NameError: name 'freq' is not defined

<Figure size 800x600 with 0 Axes>

In [9]:
# -------------------------
# Aplicar KMeans
# -------------------------
kmeans = KMeans(n_clusters=k_ideal, n_init=20, random_state=10)
clusters = kmeans.fit_predict(dados_padronizados)

NameError: name 'k_ideal' is not defined

In [10]:
# -------------------------
# Visualizar clusters
# -------------------------

visualizer = KElbowVisualizer(kmeans, k=range(3, 10), force_model=True)

visualizer.fit(dados_padronizados)
visualizer.show()


NameError: name 'KElbowVisualizer' is not defined

In [11]:
# -------------------------
# Visualizar clusters com PCA (2D)
# -------------------------
pca = PCA(n_components=2)
dados_pca = pca.fit_transform(dados_padronizados)

plt.figure(figsize=(8,6))
sns.scatterplot(x=dados_pca[:,0], y=dados_pca[:,1], hue=clusters, palette='Set2', s=60)
plt.title(f"Clusters KMeans com k = {k_ideal}")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(title='Cluster')
plt.show()

NameError: name 'clusters' is not defined

<Figure size 800x600 with 0 Axes>

In [12]:
silhouette_avg = silhouette_score(dados_padronizados, clusters)
print("Silhouette média:", silhouette_avg)

NameError: name 'clusters' is not defined

In [13]:
# -------------------------
# Silhouette
# -------------------------

silhouette_vals = silhouette_samples(dados_padronizados, clusters)

y_lower = 10

for i in range(k_ideal):
    cluster_vals = silhouette_vals[clusters == i]
    cluster_vals.sort()
    
    size_cluster = cluster_vals.shape[0]
    y_upper = y_lower + size_cluster
    
    plt.fill_betweenx(np.arange(y_lower, y_upper),
                      0, cluster_vals)
    
    plt.text(-0.05, y_lower +   0.5 * size_cluster, str(i))
    y_lower = y_upper + 10

# Plot silhouette
plt.title("Silhouette - KMeans")
plt.xlabel("Coeficiente de Silhueta")
plt.ylabel("Cluster")
plt.show()

NameError: name 'clusters' is not defined

In [14]:
# -------------------------
# Tamanho dos clusters
# -------------------------

unique, counts = np.unique(clusters, return_counts=True)
cluster_sizes = dict(zip(unique, counts))

print("Tamanho dos clusters:")
print(cluster_sizes)

NameError: name 'clusters' is not defined